In [2]:
import os, pickle, signal, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from tqdm import tqdm
import safe as sf

/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE = "/data/ryanschen/safe-retro"
ONMT_TRANSLATE = f"{BASE}/saferetrouv/bin/onmt_translate"
TEST_CSV = "/data/ryanschen/safe-retro/Data/test_safe.csv"

MODELS = {
    "SMILES":   f"{BASE}/smiles_run/model_step_400000.pt",
    "SAFE":     f"{BASE}/safe_run/model_step_400000.pt",
    "SAFE+Aug": f"{BASE}/safe_aug_run/model_step_400000.pt",
}
SRC_TEST = {
    "SMILES":   f"{BASE}/USPTO_SMILES_preprocessed/src-test.txt",
    "SAFE":     f"{BASE}/USPTO_SAFE_preprocessed/src-test.txt",
    "SAFE+Aug": f"{BASE}/USPTO_SAFE_preprocessed/src-test.txt",
}
PRED_FILES = {
    "SMILES":   f"{BASE}/smiles_run/predictions.txt",
    "SAFE":     f"{BASE}/safe_run/predictions.txt",
    "SAFE+Aug": f"{BASE}/safe_aug_run/predictions.txt",
}
CACHE_FILES = {
    "SMILES":   f"{BASE}/smiles_run/eval_cache.pkl",
    "SAFE":     f"{BASE}/safe_run/eval_cache.pkl",
    "SAFE+Aug": f"{BASE}/safe_aug_run/eval_cache.pkl",
}
GT_FILES = {
    "SMILES":   f"{BASE}/USPTO_SMILES_preprocessed/tgt-test.txt",
    "SAFE":     f"{BASE}/USPTO_SAFE_preprocessed/tgt-test.txt",
    "SAFE+Aug": f"{BASE}/USPTO_SAFE_preprocessed/tgt-test.txt",
}

SAFE_MODELS    = {"SAFE", "SAFE+Aug"}
N_BEST         = 10
BEAM_SIZE      = 10
GPU            = 0
DECODE_TIMEOUT = 5
COLORS = {"SMILES": "#4C72B0", "SAFE": "#DD8452", "SAFE+Aug": "#55A868"}

test_df = pd.read_csv(TEST_CSV)
print(f"Test set: {len(test_df):,} reactions | columns: {list(test_df.columns)}")

Test set: 39,994 reactions | columns: ['precursors', 'products', 'split', 'rxn_smiles', 'safe']


In [3]:
# Check Files Exist
all_ok = True
for name in MODELS:
    for label, path in [("model", MODELS[name]), ("src-test", SRC_TEST[name])]:
        ok = os.path.exists(path)
        print(f"  {'OKKKK' if ok else 'MISSING'} [{name}] {label}: {path}")
        all_ok = all_ok and ok

ok_csv = os.path.exists(TEST_CSV)
print(f"  {'OKKKK' if ok_csv else 'MISSING'} test.csv: {TEST_CSV}")

if all_ok and ok_csv:
    print("Can now run inference")
else:
    print("Stuff is missing")

  OKKKK [SMILES] model: /data/ryanschen/safe-retro/smiles_run/model_step_400000.pt
  OKKKK [SMILES] src-test: /data/ryanschen/safe-retro/USPTO_SMILES_preprocessed/src-test.txt
  OKKKK [SAFE] model: /data/ryanschen/safe-retro/safe_run/model_step_400000.pt
  OKKKK [SAFE] src-test: /data/ryanschen/safe-retro/USPTO_SAFE_preprocessed/src-test.txt
  OKKKK [SAFE+Aug] model: /data/ryanschen/safe-retro/safe_aug_run/model_step_400000.pt
  OKKKK [SAFE+Aug] src-test: /data/ryanschen/safe-retro/USPTO_SAFE_preprocessed/src-test.txt
  OKKKK test.csv: /data/ryanschen/safe-retro/Data/test_safe.csv
Can now run inference


# Results of the Model (SAFE Augmented and SMILES Baseline)

In [4]:
import os

# delete caches
for name in MODELS:
    if os.path.exists(CACHE_FILES[name]):
        os.remove(CACHE_FILES[name])
        print(f"Deleted cache: {CACHE_FILES[name]}")

# clear in-memory state
for var in ["decoded", "results"]:
    if var in dir():
        del globals()[var]
        print(f"Cleared: {var}")

print("\nAll clean.")


All clean.


In [5]:
# Decoding Helpers

def _timeout_handler(signum, frame):
    raise TimeoutError()

def to_canonical_smiles(smi: str) -> str:
    try:
        smi = smi.strip().replace(" ", "")
        parts = smi.split(".")
        canon_parts = []
        for p in parts:
            mol = Chem.MolFromSmiles(p)
            if mol:
                canon_parts.append(Chem.MolToSmiles(mol))
        return ".".join(sorted(canon_parts)) if canon_parts else ""
    except Exception:
        return ""

def safe_to_canonical(safe_str: str) -> str:
    try:
        safe_str = safe_str.strip().replace(" ", "")
        molecules = safe_str.split("~")
        canon_parts = []
        for mol_safe in molecules:
            signal.signal(signal.SIGALRM, _timeout_handler)
            signal.alarm(DECODE_TIMEOUT)
            try:
                mol = sf.decode(mol_safe, as_mol=True, ignore_errors=True)
                signal.alarm(0)
                if mol is not None:
                    canon_parts.append(Chem.MolToSmiles(mol))
            except Exception:
                signal.alarm(0)
                continue
        return ".".join(sorted(canon_parts)) if canon_parts else ""
    except Exception:
        return ""

print("Decoders defined.")

Decoders defined.


In [6]:
GT_FILES = {
    "SMILES":   f"{BASE}/USPTO_SMILES_preprocessed/tgt-test.txt",
    "SAFE":     f"{BASE}/USPTO_SAFE_preprocessed/tgt-test.txt",
    "SAFE+Aug": f"{BASE}/USPTO_SAFE_preprocessed/tgt-test.txt",
}

def decode_predictions(name):
    cache = CACHE_FILES[name]
    if os.path.exists(cache):
        print(f"[{name}] Loading from cache...")
        with open(cache, "rb") as f:
            return pickle.load(f)

    if not os.path.exists(PRED_FILES[name]):
        print(f"[{name}] Predictions not ready yet — skipping")
        return None

    decoder = safe_to_canonical if name in SAFE_MODELS else to_canonical_smiles

    with open(PRED_FILES[name]) as f:
        raw = [line.strip() for line in f]
    with open(GT_FILES[name]) as f:
        gt_raw = [line.strip() for line in f]

    n_actual = min(len(raw) // N_BEST, len(gt_raw))
    print(f"[{name}] Evaluating {n_actual:,} reactions")

    grouped = [raw[i * N_BEST:(i+1) * N_BEST] for i in range(n_actual)]

    print(f"[{name}] Decoding predictions...")
    pred_smiles = [[decoder(p) for p in preds] for preds in tqdm(grouped, desc=name)]

    print(f"[{name}] Decoding ground truth...")
    gt_smiles = [decoder(g) for g in tqdm(gt_raw[:n_actual], desc="GT")]

    data = {"pred_smiles": pred_smiles, "gt_smiles": gt_smiles, "n": n_actual}
    with open(cache, "wb") as f:
        pickle.dump(data, f)
    print(f"[{name}] Cached to {cache}")
    return data

decoded = {}
for name in MODELS:
    decoded[name] = decode_predictions(name)
    print()

[SMILES] Evaluating 39,994 reactions
[SMILES] Decoding predictions...


SMILES:   0%|          | 0/39994 [00:00<?, ?it/s][16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c2c32)CC1O'
[16:04:45] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c23)CC1O'
[16:04:45] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c23)CC1O'
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WARNING: not removing hydrogen atom without neighbors
[16:04:45] WAR

[SMILES] Decoding ground truth...


GT:   0%|          | 0/39994 [00:00<?, ?it/s][16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen atom without neighbors
[16:05:45] WARNING: not removing hydrogen ato

[SMILES] Cached to /data/ryanschen/safe-retro/smiles_run/eval_cache.pkl

[SAFE] Evaluating 39,994 reactions
[SAFE] Decoding predictions...


...

[SAFE] Cached to /data/ryanschen/safe-retro/safe_run/eval_cache.pkl

[SAFE+Aug] Evaluating 39,994 reactions
[SAFE+Aug] Decoding predictions...


SAFE+Aug: 100%|██████████| 39994/39994 [04:38<00:00, 143.78it/s]


[SAFE+Aug] Decoding ground truth...


GT: 100%|██████████| 39994/39994 [00:30<00:00, 1322.14it/s]

[SAFE+Aug] Cached to /data/ryanschen/safe-retro/safe_aug_run/eval_cache.pkl



In [7]:
# Verification of GT before Calculating Metrics
for name in [n for n in MODELS if decoded.get(n) is not None]:
    print(f"\n=== {name} ===")
    for i in range(2):
        gt = decoded[name]["gt_smiles"][i]
        p1 = decoded[name]["pred_smiles"][i][0]
        print(f"GT  : {gt}")
        print(f"P1  : {p1}")
        print(f"Match: {gt == p1}")


=== SMILES ===
GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
Match: False
GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
P1  : C1CCOC1.CCOC(=O)N=NC(=O)OCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO.c1ccc(P(c2ccccc2)c2ccccc2)cc1
Match: False

=== SAFE ===
GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
Match: False
GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
P1  : C1CCOC1.CC(C)OC(=O)N=NC(=O)OC(C)C.COC(=O)Cc1cn(C)c2cc(O)ccc12
Match: False

=== SAFE+Aug ===
GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
Match: False
GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
P1  : CN(C)C=O.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccc(-c2cn(C(F)(F)F)nc2C)

In [8]:
# Remove spaces and see if it's valid SMILES
raw = "N 1 2 3 . C C 1 . C 2 C . C 3 C"
smi = raw.replace(" ", "")
print("Joined:", smi)
mol = Chem.MolFromSmiles(smi)
print("Valid SMILES:", mol is not None)
if mol:
    print("Canonical:", Chem.MolToSmiles(mol))

Joined: N123.CC1.C2C.C3C
Valid SMILES: True
Canonical: CCN(CC)CC


In [9]:
# Taking off Reagents and Calculating Top-k Accuracy

def topk_accuracy(pred_list, gt_list, k):
    correct = sum(gt in preds[:k] for gt, preds in zip(gt_list, pred_list))
    return correct / len(gt_list)

def coverage(pred_list):
    return sum(any(p for p in preds) for preds in pred_list) / len(pred_list)

def valid_rate(pred_list):
    total = sum(len(p) for p in pred_list)
    valid = sum(1 for preds in pred_list for p in preds if p)
    return valid / total

_REAGENTS_EXTENDED = {
    # solvents
    "C1CCOC1", "CCO", "CO", "CCOC", "CC(C)O", "CC(O)=O", "CC#N",
    "ClCCl", "ClC(Cl)Cl", "c1ccccc1", "Cc1ccccc1", "CCOCC", "O",
    "CN(C)C=O", "CS(C)=O", "C1CCNCC1", "c1ccncc1", "Cc1ccccn1",
    # bases / acids
    "[Na+].[OH-]", "[K+].[OH-]", "CC(C)(C)[O-].[Na+]",
    "[O-]C(=O)[O-].[Na+].[Na+]", "O=C([O-])[O-].[K+].[K+]",
    # simple salts / atoms
    "[H-]", "[Na+]", "[K+]", "[Li+]", "Cl", "Br", "[F-]",
    "[NH4+].[Cl-]", "[H][H]", "O=O", "N",
    # common acids
    "O=S(=O)(O)O", "Cl.O", "OO",
}

_METALS = {"Li","Na","K","Cs","Mg","Ca","Al","Zn","Fe","Cu",
           "Pd","Ni","Rh","Ir","Ru","Os","Pt","Au","Ag","B"}

def strip_reagents(smi: str) -> str:
    if not smi:
        return ""
    parts = smi.split(".")
    core = []
    for p in parts:
        if p in _REAGENTS_EXTENDED:
            continue
        mol = Chem.MolFromSmiles(p)
        if mol is None:
            continue
        atoms = {a.GetSymbol() for a in mol.GetAtoms()}
        # skip pure metal/inorganic fragments
        if atoms & _METALS and not (atoms - _METALS - {"C","H","O","N","Cl","F","Br"}):
            continue
        # skip very small fragments (1-2 heavy atoms) — likely salts/counterions
        if mol.GetNumHeavyAtoms() <= 2:
            continue
        core.append(Chem.MolToSmiles(mol))
    return ".".join(sorted(core)) if core else smi

In [10]:
# Computation of Metrics

results = {}
for name in MODELS:
    if decoded.get(name) is None:
        print(f"[{name}] Skipped (no predictions)")
        continue
    pred = decoded[name]["pred_smiles"]
    gt   = decoded[name]["gt_smiles"]
    pred_core = [[strip_reagents(p) for p in preds] for preds in pred]
    gt_core   = [strip_reagents(g) for g in gt]
    results[name] = {
        "exact":      {k: topk_accuracy(pred, gt, k) for k in [1,3,5,10]},
        "core":       {k: topk_accuracy(pred_core, gt_core, k) for k in [1,3,5,10]},
        "coverage":   coverage(pred),
        "valid_rate": valid_rate(pred),
        "pred":       pred,
        "gt":         gt,
        "pred_core":  pred_core,
        "gt_core":    gt_core,
    }
print("Done.")

[16:15:58] WARNING: not removing hydrogen atom without neighbors
[16:15:59] WARNING: not removing hydrogen atom without neighbors
[16:15:59] WARNING: not removing hydrogen atom without neighbors
[16:16:09] WARNING: not removing hydrogen atom without neighbors
[16:16:10] WARNING: not removing hydrogen atom without neighbors
[16:16:10] WARNING: not removing hydrogen atom without neighbors
[16:16:10] WARNING: not removing hydrogen atom without neighbors
[16:16:10] WARNING: not removing hydrogen atom without neighbors
[16:16:12] WARNING: not removing hydrogen atom without neighbors
[16:16:12] WARNING: not removing hydrogen atom without neighbors
[16:16:14] WARNING: not removing hydrogen atom without neighbors
[16:16:16] WARNING: not removing hydrogen atom without neighbors
[16:16:16] WARNING: not removing hydrogen atom without neighbors
[16:16:16] WARNING: not removing hydrogen atom without neighbors
[16:16:16] WARNING: not removing hydrogen atom without neighbors
[16:16:22] WARNING: not r

Done.


In [11]:
# Print Results

names = [n for n in MODELS if n in results]
col = 14
label_w = 26

print(f"\n{'Metric':<{label_w}}", end="")
for name in names:
    print(f"{name:>{col}}", end="")
print()
print("-" * (label_w + col * len(names)))

for k in [1, 3, 5, 10]:
    print(f"{'Top-'+str(k)+' exact':<{label_w}}", end="")
    for name in names:
        print(f"{results[name]['exact'][k]*100:>{col-1}.2f}%", end="")
    print()

print()
for k in [1, 3, 5, 10]:
    print(f"{'Top-'+str(k)+' core-only':<{label_w}}", end="")
    for name in names:
        print(f"{results[name]['core'][k]*100:>{col-1}.2f}%", end="")
    print()

print()
print(f"{'Coverage':<{label_w}}", end="")
for name in names:
    print(f"{results[name]['coverage']*100:>{col-1}.2f}%", end="")
print()
print(f"{'Valid rate':<{label_w}}", end="")
for name in names:
    print(f"{results[name]['valid_rate']*100:>{col-1}.2f}%", end="")
print()


Metric                            SMILES          SAFE      SAFE+Aug
--------------------------------------------------------------------
Top-1 exact                       18.31%         4.72%         1.44%
Top-3 exact                       26.73%         7.60%         2.38%
Top-5 exact                       30.09%         8.84%         2.83%
Top-10 exact                      33.43%        10.10%         3.25%

Top-1 core-only                   32.15%        11.53%         4.82%
Top-3 core-only                   43.02%        15.92%         6.63%
Top-5 core-only                   47.05%        17.63%         7.37%
Top-10 core-only                  51.01%        19.20%         8.07%

Coverage                         100.00%        99.99%        99.99%
Valid rate                        99.99%        99.96%        99.93%


In [12]:
# Raw SAFE string match vs Canonical SMILES match
print("SAFE Raw String vs Canonical SMILES comparison")
print(f"{'Metric':<30} {'Raw SAFE':>12} {'Canonical':>12}")
print("-" * 55)

for name in [n for n in MODELS if n in SAFE_MODELS and n in results]:
    with open(PRED_FILES[name]) as f:
        raw_preds = [l.strip().replace(" ", "") for l in f]
    with open(GT_FILES[name]) as f:
        raw_gt = [l.strip().replace(" ", "") for l in f]

    n = len(raw_gt)
    grouped_raw = [raw_preds[i*N_BEST:(i+1)*N_BEST] for i in range(n)]

    for k in [1, 10]:
        raw_match = sum(gt in preds[:k] for gt, preds in zip(raw_gt, grouped_raw)) / n
        canon_match = results[name]["exact"][k]
        print(f"{name} Top-{k} {'raw':<20} {raw_match*100:>11.2f}% {canon_match*100:>11.2f}%")
    print()


SAFE Raw String vs Canonical SMILES comparison
Metric                             Raw SAFE    Canonical
-------------------------------------------------------
SAFE Top-1 raw                         4.13%        4.72%
SAFE Top-10 raw                         9.15%       10.10%

SAFE+Aug Top-1 raw                         0.64%        1.44%
SAFE+Aug Top-10 raw                         1.92%        3.25%



In [13]:
# Clear all caches
for name in MODELS:
    if os.path.exists(CACHE_FILES[name]):
        os.remove(CACHE_FILES[name])
        print(f"Deleted: {CACHE_FILES[name]}")

Deleted: /data/ryanschen/safe-retro/smiles_run/eval_cache.pkl
Deleted: /data/ryanschen/safe-retro/safe_run/eval_cache.pkl
Deleted: /data/ryanschen/safe-retro/safe_aug_run/eval_cache.pkl


In [14]:
# Spot-check first 2 predictions for each evaluated model
for name in [n for n in MODELS if decoded.get(n) is not None]:
    print(f"=== {name} ===")
    for i in range(2):
        gt = decoded[name]["gt_smiles"][i]
        p1 = decoded[name]["pred_smiles"][i][0]
        print(f"  GT  : {gt}")
        print(f"  P1  : {p1}")
        print(f"  Match: {gt == p1}")
    print()

=== SMILES ===
  GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
  P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
  Match: False
  GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
  P1  : C1CCOC1.CCOC(=O)N=NC(=O)OCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO.c1ccc(P(c2ccccc2)c2ccccc2)cc1
  Match: False

=== SAFE ===
  GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
  P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
  Match: False
  GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
  P1  : C1CCOC1.CC(C)OC(=O)N=NC(=O)OC(C)C.COC(=O)Cc1cn(C)c2cc(O)ccc12
  Match: False

=== SAFE+Aug ===
  GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
  P1  : N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F
  Match: False
  GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
  P1  : CN(C)C=O.COC(=O)Cc1cn(C)c2cc(O)

In [15]:
# compare first line of both test src files
with open("/data/ryanschen/safe-retro/USPTO_SMILES_preprocessed/src-test.txt") as f:
    smiles_src = f.readline().strip()

with open("/data/ryanschen/safe-retro/USPTO_SAFE_preprocessed/src-test.txt") as f:
    safe_src = f.readline().strip()

print("SMILES src:", smiles_src)
print("SAFE src  :", safe_src)

print("GT precursors:", test_df["precursors"].iloc[0])
print("GT safe col  :", test_df["safe"].iloc[0])

SMILES src: N # C c 1 c c s c 1 N c 1 c c ( F ) c ( F ) c c 1 [N+] ( = O ) [O-]
SAFE src  : c 1 3 c c ( F ) c ( F ) c c 1 [N+] ( = O ) [O-] . N # C c 1 c c s c 1 2 . N 2 3
GT precursors: C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
GT safe col  : C1CCOC1~N#Cc1ccsc1N~O=[N+]([O-])c1cc(F)c(F)cc1F~[H-]~[Na+]>>c13cc(F)c(F)cc1[N+](=O)[O-].N#Cc1ccsc12.N23


In [4]:
# Per-class accuracy using existing cluster labels
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

# Load fingerprints and recompute clusters (or load saved labels)
all_fps = np.load(f"{BASE}/test_rxnfp.npy")
fps_norm = normalize(all_fps)
kmeans = MiniBatchKMeans(n_clusters=10, random_state=42, n_init=10, batch_size=1024)
cluster_labels = kmeans.fit_predict(fps_norm)

cluster_names = {
    0: "Deprotection / Ether cleavage",
    1: "O-Alkylation / Heteroatom alkylation",
    2: "Oxidation",
    3: "Reduction (carbonyl)",
    4: "N-Arylation / C-N coupling",
    5: "Acylation / Amide formation",
    6: "Protection (Boc)",
    7: "C-C Bond formation",
    8: "Reduction (nitro→amine)",
    9: "O-Alkylation / Ester formation",
}

# Compute correctness flags per model from decoded dict
model_correct = {}
for name in [n for n in MODELS if n in results]:
    pred = decoded[name]["pred_smiles"]
    gt   = decoded[name]["gt_smiles"]
    model_correct[name] = {
        "top1":  [g != "" and g == p[0]  for g, p in zip(gt, pred)],
        "top3":  [g != "" and g in p[:3] for g, p in zip(gt, pred)],
        "top10": [g != "" and g in p     for g, p in zip(gt, pred)],
    }

# Print per-class table
names = list(model_correct.keys())
col = 10
print(f"\n{'Cluster':<6} {'Reaction Type':<35} {'N':>5}", end="")
for name in names:
    print(f"  {name+' T1':>{col}} {name+' T10':>{col}}", end="")
print()
print("-" * (46 + (col*2 + 2) * len(names)))

for cls in sorted(set(cluster_labels)):
    idx = [i for i, c in enumerate(cluster_labels) if c == cls]
    n = len(idx)
    print(f"{cls:<6} {cluster_names[cls]:<35} {n:>5}", end="")
    for name in names:
        t1  = sum(model_correct[name]["top1"][i]  for i in idx) / n * 100
        t10 = sum(model_correct[name]["top10"][i] for i in idx) / n * 100
        print(f"  {t1:>{col-1}.1f}% {t10:>{col-1}.1f}%", end="")
    print()

print(f"\n{'Overall':<42} {len(cluster_labels):>5}", end="")
for name in names:
    t1  = sum(model_correct[name]["top1"])  / len(cluster_labels) * 100
    t10 = sum(model_correct[name]["top10"]) / len(cluster_labels) * 100
    print(f"  {t1:>{col-1}.1f}% {t10:>{col-1}.1f}%", end="")
print()


NameError: name 'results' is not defined